In [2]:
import pandas as pd
import yfinance as yf
from pathlib import Path

In [3]:
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
tickers = {
    "BTC": "BTC-USD",
    "ETH": "ETH-USD",
    "NASDAQ": "^NDX",
    "SP500": "^GSPC",
    "GOLD": "GC=F",
    "DXY": "DX-Y.NYB",
    "US10Y": "^TNX"
}

In [5]:
start_date = "2018-01-01"
end_date = None

data = yf.download(
    list(tickers.values()),
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)

data.head()

Price              Close                                                   \
Ticker           BTC-USD   DX-Y.NYB     ETH-USD         GC=F        ^GSPC   
Date                                                                        
2018-01-01  13657.200195        NaN  772.640991          NaN          NaN   
2018-01-02  14982.099609  91.849998  884.443970  1313.699951  2695.810059   
2018-01-03  15201.000000  92.160004  962.719971  1316.199951  2713.060059   
2018-01-04  15599.200195  91.849998  980.921997  1319.400024  2723.989990   
2018-01-05  17429.500000  91.949997  997.719971  1320.300049  2743.149902   

Price                                   High                          ...  \
Ticker             ^NDX   ^TNX       BTC-USD   DX-Y.NYB      ETH-USD  ...   
Date                                                                  ...   
2018-01-01          NaN    NaN  14112.200195        NaN   782.530029  ...   
2018-01-02  6511.339844  2.465  15444.599609  92.250000   914.830017  ...   
2018-01-03  6575.799805  2.447  15572.799805  92.260002   974.471008  ...   
2018-01-04  6584.580078  2.453  15739.700195  92.260002  1045.079956  ...   
2018-01-05  6653.290039  2.476  17705.199219  92.099998  1075.390015  ...   

Price              Open                           Volume                       \
Ticker            ^GSPC         ^NDX   ^TNX      BTC-USD DX-Y.NYB     ETH-USD   
Date                                                                            
2018-01-01          NaN          NaN    NaN  10291200000      NaN  2595760128   
2018-01-02  2683.729980  6431.589844  2.433  16846600192      0.0  5783349760   
2018-01-03  2697.850098  6520.029785  2.451  16871900160      0.0  5093159936   
2018-01-04  2719.310059  6595.750000  2.473  21783199744      0.0  6502859776   
2018-01-05  2731.330078  6613.129883  2.465  23840899072      0.0  6683149824   

Price                                              
Ticker      GC=F         ^GSPC          ^NDX ^TNX  
Date                                               
2018-01-01   NaN           NaN           NaN  NaN  
2018-01-02  68.0  3.397430e+09  1.929700e+09  0.0  
2018-01-03  42.0  3.544030e+09  2.173130e+09  0.0  
2018-01-04   2.0  3.697340e+09  2.103220e+09  0.0  
2018-01-05   1.0  3.239280e+09  2.024000e+09  0.0  

[5 rows x 35 columns]

In [6]:
close_prices = data["Close"].copy()

# Rename columns from ticker symbols to readable asset names
reverse_tickers = {v: k for k, v in tickers.items()}
close_prices = close_prices.rename(columns=reverse_tickers)

close_prices.head()

Ticker,BTC,DXY,ETH,GOLD,SP500,NASDAQ,US10Y
Date,,,,,,,
2018-01-01,13657.200195,NaN,772.640991,NaN,NaN,NaN,NaN
2018-01-02,14982.099609,91.849998,884.443970,1313.699951,2695.810059,6511.339844,2.465
2018-01-03,15201.000000,92.160004,962.719971,1316.199951,2713.060059,6575.799805,2.447
2018-01-04,15599.200195,91.849998,980.921997,1319.400024,2723.989990,6584.580078,2.453
2018-01-05,17429.500000,91.949997,997.719971,1320.300049,2743.149902,6653.290039,2.476


In [7]:
missing_summary = close_prices.isna().sum().sort_values(ascending=False)
missing_summary

Ticker
SP500     956
NASDAQ    956
US10Y     956
GOLD      954
DXY       953
BTC         0
ETH         0
dtype: int64

In [8]:
close_prices_clean = close_prices.ffill()

# Drop rows where all assets are still missing
close_prices_clean = close_prices_clean.dropna(how="all")

close_prices_clean.head()

Ticker,BTC,DXY,ETH,GOLD,SP500,NASDAQ,US10Y
Date,,,,,,,
2018-01-01,13657.200195,NaN,772.640991,NaN,NaN,NaN,NaN
2018-01-02,14982.099609,91.849998,884.443970,1313.699951,2695.810059,6511.339844,2.465
2018-01-03,15201.000000,92.160004,962.719971,1316.199951,2713.060059,6575.799805,2.447
2018-01-04,15599.200195,91.849998,980.921997,1319.400024,2723.989990,6584.580078,2.453
2018-01-05,17429.500000,91.949997,997.719971,1320.300049,2743.149902,6653.290039,2.476


In [9]:
print("Shape:", close_prices_clean.shape)
print("Start date:", close_prices_clean.index.min())
print("End date:", close_prices_clean.index.max())

close_prices_clean.tail()

Shape: (3063, 7)
Start date: 2018-01-01 00:00:00
End date: 2026-05-21 00:00:00


Ticker,BTC,DXY,ETH,GOLD,SP500,NASDAQ,US10Y
Date,,,,,,,
2026-05-17,77429.351562,99.269997,2127.645264,4555.799805,7408.500000,29125.199219,4.595
2026-05-18,76954.171875,98.970001,2128.516113,4552.500000,7403.049805,28994.369141,4.623
2026-05-19,76750.906250,99.300003,2109.963867,4506.299805,7353.609863,28818.839844,4.667
2026-05-20,77457.773438,99.110001,2126.983398,4531.299805,7432.970215,29297.699219,4.572
2026-05-21,77245.617188,99.227997,2116.179932,4519.700195,7432.970215,29297.699219,4.572


In [10]:
output_path = PROCESSED_DATA_DIR / "multi_asset_prices.csv"

close_prices_clean.to_csv(output_path)

print(f"Saved to: {output_path}")

Saved to: ../data/processed/multi_asset_prices.csv


In [11]:
check_df = pd.read_csv(output_path)

check_df.head()

,Date,BTC,DXY,ETH,GOLD,SP500,NASDAQ,US10Y
0,2018-01-01,13657.200195,NaN,772.640991,NaN,NaN,NaN,NaN
1,2018-01-02,14982.099609,91.849998,884.443970,1313.699951,2695.810059,6511.339844,2.465
2,2018-01-03,15201.000000,92.160004,962.719971,1316.199951,2713.060059,6575.799805,2.447
3,2018-01-04,15599.200195,91.849998,980.921997,1319.400024,2723.989990,6584.580078,2.453
4,2018-01-05,17429.500000,91.949997,997.719971,1320.300049,2743.149902,6653.290039,2.476
